# Part 13 — MapBiomas realized use + potential-vs-realized (RQ2)

Aggregate the GEE-hosted MapBiomas collection to a 250 m realized-use image (majority class + segment role + per-role area fractions + Brazil-specific masks), then cross it with `suit_present` for the **under-utilization** map (high potential ∧ not currently in that use) and profile realized composition by biophysical zone. Engine: `src/external.py`.

**Guardrail:** MapBiomas is **mask + validation + descriptive profiling only** — it never enters the feature stack or the clusterer. The masks may *replace* the coarse Part-7b WorldCover mask for a higher-fidelity membership re-run (`external.hybrid_lc`), but that does not feed back into Parts 9–10.

**Output:** `feat_realized`, `realized_vs_potential` + zone-composition tables. **DoD:** RQ2 answered; soy/cane realized footprints concentrate in high-suitability classes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import ee, geemap
import utils, features
project = utils.init()
print('EE initialized; project =', project)
import external, membership

In [ ]:
aoi = utils.load_aoi(project)
print('AOI area (km^2):', round(aoi.area(1000).divide(1e6).getInfo(), 1))

### Build the 250 m realized-use image (majority class/role + masks + role fractions)

In [ ]:
realized = external.realized_features(aoi)
print('bands:', realized.bandNames().getInfo())

### DoD sanity — realized-role fractions and masks over AOI

In [ ]:
utils.range_report(realized, aoi)

### Potential-vs-realized — under-utilization (high potential, other realized use)

In [ ]:
suit = ee.Image(utils.asset_id(project, 'suit_present'))
segs = list(utils.cfg('segments')['segments'])
uu = external.underutilization(suit, realized, segs)
print('under-utilization bands:', uu.bandNames().getInfo())
# AOI area share flagged under-utilized per segment
import pandas as pd
shares = uu.reduceRegion(ee.Reducer.mean(), aoi, 1000, maxPixels=int(1e12),
                         bestEffort=True, tileScale=8).getInfo()
pd.Series(shares).round(3)

### Realized composition per biophysical zone (descriptive profiling)

In [ ]:
zones = ee.Image(utils.asset_id(project, 'zones_present'))
n_zones = int(zones.reduceRegion(ee.Reducer.max(), aoi, 1000, maxPixels=int(1e12),
              bestEffort=True).values().get(0).getInfo()) + 1
rows = []
for z in range(n_zones):
    h = (realized.select('rl_role').updateMask(zones.eq(z))
         .reduceRegion(ee.Reducer.frequencyHistogram(), aoi, 1000,
                       maxPixels=int(1e12), bestEffort=True, tileScale=8)
         .getInfo()['rl_role'])
    tot = sum(h.values())
    rows.append({'zone': z, **{f'role_{k}': round(v / tot, 3) for k, v in h.items()}})
pd.DataFrame(rows).fillna(0)

### Quick look — realized role + a soybean under-utilization map

In [ ]:
Map = geemap.Map(); Map.centerObject(aoi, 7)
role_pal = ['#eeeeee','#ffd400','#7b3294','#d95f0e','#2c7fb8','#addd8e','#006837']
Map.addLayer(realized.select('rl_role'), {'min': 0, 'max': 6, 'palette': role_pal}, 'realized role')
Map.addLayer(uu.select('underused_soybean'), {'min': 0, 'max': 1, 'palette': ['white','red']}, 'under-used soybean', False)
Map.addLayer(aoi, {}, 'AOI', False)
Map

### Export `feat_realized` + `realized_vs_potential`

In [ ]:
utils.ensure_folder(project)
t1 = utils.export_image(realized, project, 'feat_realized', aoi)
t2 = utils.export_image(uu.toByte(), project, 'realized_vs_potential', aoi)
print(utils.task_summary([t1, t2]))